In [2]:
# =============================================================================
#  Error Mitigation Baseline — IBM Quantum Task 4
#  Probabilistic YK Decoder (Implementation A, post-selected)
#
#  Four conditions compared on ibm_torino:
#    1. Unmitigated          — raw SamplerV2, no mitigation
#    2. DD only              — dynamical decoupling (XX sequence) enabled
#    3. Readout-mitigated    — mthree calibration matrix applied post-hoc
#    4. ZNE (manual)         — gate folding at λ=1,3,5 + linear extrapolation
#
#  Note on ZNE: SamplerV2 does not expose ZNE natively (Estimator-only in
#  Qiskit Runtime). We implement digital ZNE manually by folding two-qubit
#  gates (CZ → CZ·CZ†·CZ for λ=3, CZ·CZ†·CZ·CZ†·CZ for λ=5) and
#  extrapolating F to λ=0.
#
#  Outputs : error_mitigation_results.json
#            (pass to plot_error_mitigation.py for figures)
# =============================================================================

import numpy as np
import warnings, json, datetime
warnings.filterwarnings("ignore")
from collections import Counter, defaultdict
from scipy.stats import beta as beta_dist

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector, DensityMatrix, state_fidelity
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler, Batch

# =============================================================================
#  CONFIG
# =============================================================================
IBM_TOKEN   = "QfkScNfX4bVJ5lXm0082x7F7J6vya3SF5LJZFNOYXqqO"
BACKEND     = "ibm_torino"
SHOTS       = 10000
N_BOOTSTRAP = 2000
SEED        = 42
RNG         = np.random.default_rng(SEED)
THETA_MSG   = 2.5349076035276403
VARPHI_MSG  = 2.0022404587009195
ZNE_FACTORS = [1, 3, 5]      # noise amplification factors for manual ZNE
DD_SEQUENCE = "XX"           # dynamical decoupling sequence type

# =============================================================================
#  STEP 1 — Core circuit builder
# =============================================================================
def build_tomo_circuit(basis='Z', noise_factor=1,
                       theta=THETA_MSG, varphi=VARPHI_MSG):
    """
    Build a single tomography circuit for the post-selected YK decoder.

    noise_factor : int, odd — ZNE gate folding level.
        1 = no folding (standard circuit)
        3 = each CZ replaced by CZ·CZ†·CZ
        5 = each CZ replaced by CZ·CZ†·CZ·CZ†·CZ

    The circuit measures C (output register) in the given Pauli basis,
    with crR and crG available for post-selection.
    """
    C  = QuantumRegister(1,'C');  E  = QuantumRegister(1,'E')
    R  = QuantumRegister(1,'R');  G  = QuantumRegister(1,'G')
    M  = QuantumRegister(1,'M');  A  = QuantumRegister(1,'A')
    Yr = QuantumRegister(1,'Y')
    crR   = ClassicalRegister(1,'crR')
    crG   = ClassicalRegister(1,'crG')
    crT   = ClassicalRegister(1,'crTomo')
    qc    = QuantumCircuit(C,E,R,G,M,A,Yr,crR,crG,crT)

    # ── Message preparation ───────────────────────────────────────────────────
    qc.u(theta, varphi, 0.0, M)
    qc.swap(C, M); qc.barrier()

    # ── Bell pairs ────────────────────────────────────────────────────────────
    qc.h(E);  qc.cx(E, M)
    qc.h(R);  qc.cx(R, G)
    qc.h(A);  qc.cx(A, Yr); qc.barrier()

    # ── Scrambling unitary (with optional gate folding) ───────────────────────
    def cz_folded(q0, q1):
        """Apply CZ folded to the given noise factor."""
        for k in range(noise_factor):
            qc.cz(q0, q1)
            if k < noise_factor - 1:
                qc.cz(q0, q1)   # CZ† = CZ for CZ gate

    cz_folded(C, R);  cz_folded(E, R);  cz_folded(C, E)
    qc.h(C); qc.h(E); qc.h(R)
    cz_folded(C, R);  cz_folded(C, E);  cz_folded(E, R)
    qc.barrier()

    # ── Decoder U† (with optional gate folding) ───────────────────────────────
    cz_folded(A, G);  cz_folded(M, A);  cz_folded(G, M)
    qc.h(A); qc.h(M); qc.h(G)
    cz_folded(A, G);  cz_folded(G, M);  cz_folded(M, A)
    qc.barrier()

    # ── Bell projection ───────────────────────────────────────────────────────
    qc.cx(R, G); qc.h(R); qc.barrier()

    # ── Output: SWAP, no U† (tomography target) ───────────────────────────────
    qc.swap(C, Yr)

    # ── Measurements ─────────────────────────────────────────────────────────
    qc.measure(R, crR); qc.measure(G, crG)
    if basis == 'X':   qc.h(C)
    elif basis == 'Y': qc.sdg(C); qc.h(C)
    qc.measure(C, crT)

    return qc

# =============================================================================
#  STEP 2 — Build all circuits:
#    Unmitigated   : λ=1, no DD  (3 circuits)
#    DD            : λ=1, DD on  (3 circuits)
#    ZNE λ=3       : λ=3, no DD  (3 circuits)
#    ZNE λ=5       : λ=5, no DD  (3 circuits)
#    Total         : 12 circuits
# =============================================================================
circuit_specs = []
for basis in ['Z', 'X', 'Y']:
    circuit_specs.append(('unmitigated', basis, 1, False))
    circuit_specs.append(('dd',          basis, 1, True ))
    circuit_specs.append(('zne_3',       basis, 3, False))
    circuit_specs.append(('zne_5',       basis, 5, False))

# Readout mitigation is applied in post-processing using mthree, so no extra
# circuits are needed for it — mthree just needs the calibration data.

# =============================================================================
#  STEP 3 — Connect, transpile all 12 circuits
# =============================================================================
print("=" * 65)
print("  Error Mitigation Baseline — ibm_torino")
print("=" * 65)

service = QiskitRuntimeService(channel="ibm_quantum_platform", token=IBM_TOKEN)
backend = service.backend(BACKEND)
print(f"\n[✓] Connected to {BACKEND}")

print(f"\n  Transpiling {len(circuit_specs)} circuits ...")
transpiled = []
for cond, basis, lam, use_dd in circuit_specs:
    qc = build_tomo_circuit(basis=basis, noise_factor=lam)
    t  = transpile(qc, backend=backend,
                   optimization_level=3, seed_transpiler=SEED)
    depth = t.depth()
    twoq  = sum(1 for _,qa,_ in t.data if len(qa)==2)
    transpiled.append((cond, basis, lam, use_dd, t))
    print(f"    {cond:12s} {basis}  λ={lam}  DD={'Y' if use_dd else 'N'}"
          f"  depth={depth:4d}  2q={twoq:3d}")

# =============================================================================
#  STEP 4 — Identify physical qubits for mthree calibration
# =============================================================================
# Get the physical qubits used by the unmitigated Z-basis circuit
ref_circ = transpiled[0][4]   # unmitigated, Z basis
used_qubits = sorted(set(
    ref_circ.find_bit(q).index
    for _, qargs, _ in ref_circ.data
    for q in qargs
))
print(f"\n  Physical qubits used: {used_qubits}")

# =============================================================================
#  STEP 5 — Submit all circuits in one Batch
# =============================================================================
print(f"\n  Submitting {len(transpiled)} circuits × {SHOTS} shots (Batch) ...")

jobs = {}
with Batch(backend=backend) as batch:
    # Unmitigated and ZNE circuits — no DD
    sampler_plain = Sampler(mode=batch)
    sampler_plain.options.dynamical_decoupling.enable = False

    # DD circuits
    sampler_dd = Sampler(mode=batch)
    sampler_dd.options.dynamical_decoupling.enable = True
    sampler_dd.options.dynamical_decoupling.sequence_type = DD_SEQUENCE

    for cond, basis, lam, use_dd, t in transpiled:
        key = f"{cond}_{basis}"
        sampler = sampler_dd if use_dd else sampler_plain
        jobs[key] = sampler.run([t], shots=SHOTS)
        print(f"    {key}: job_id={jobs[key].job_id()}")

# =============================================================================
#  STEP 6 — Collect raw counts
# =============================================================================
def extract_counts(result):
    pub  = result[0]; data = pub.data
    crT  = data.crTomo.array.flatten()
    crG  = data.crG.array.flatten()
    crR  = data.crR.array.flatten()
    joint = [f"{int(t)}{int(g)}{int(r)}"
             for t,g,r in zip(crT,crG,crR)]
    return dict(Counter(joint))

print("\n  Collecting results ...")
raw = {}
for cond, basis, lam, use_dd, _ in transpiled:
    key = f"{cond}_{basis}"
    print(f"    Waiting for {key} ...", end=" ", flush=True)
    result  = jobs[key].result()
    raw[key] = extract_counts(result)
    total   = sum(raw[key].values())
    try:
        qs = jobs[key].metrics()['usage']['quantum_seconds']
        print(f"done ({total} counts, QPU={qs:.4f}s)")
    except Exception:
        print(f"done ({total} counts)")

# =============================================================================
#  STEP 7 — mthree readout mitigation calibration
#
#  mthree builds a sparse calibration matrix over the used physical qubits
#  by running "0...0" and "1...1" calibration circuits and measuring the
#  actual flip probabilities. It then inverts the matrix to correct counts.
# =============================================================================
print("\n  Running mthree readout calibration ...")
try:
    import mthree
    mit   = mthree.M3Mitigation(backend)
    mit.cals_from_system(used_qubits, shots=SHOTS)
    print(f"  [✓] mthree calibrated on qubits {used_qubits}")
    MTHREE_AVAILABLE = True
except ImportError:
    print("  [!] mthree not installed — skipping readout mitigation")
    print("      Install with: pip install mthree")
    MTHREE_AVAILABLE = False

# =============================================================================
#  STEP 8 — Post-processing helpers
# =============================================================================
def parse_bs(bs):
    p = bs.split(' ') if ' ' in bs else list(bs)
    return p[0], p[1], p[2]   # crTomo, crG, crR

def postselect_00(counts):
    kept=defaultdict(int); nt=0; nk=0
    for bs,cnt in counts.items():
        nt+=cnt
        crT,crG,crR=parse_bs(bs)
        if crR=='0' and crG=='0':
            kept[crT]+=cnt; nk+=cnt
    return dict(kept), nt, nk

def pauli_exp(c):
    n0=c.get('0',0); n1=c.get('1',0); N=n0+n1
    return (n0-n1)/N if N>0 else 0.0

def reconstruct(sx,sy,sz):
    X=np.array([[0,1],[1,0]],dtype=complex)
    Y=np.array([[0,-1j],[1j,0]],dtype=complex)
    Z=np.array([[1,0],[0,-1]],dtype=complex)
    rho=(np.eye(2)+sx*X+sy*Y+sz*Z)/2
    ev,evec=np.linalg.eigh(rho)
    ev=np.maximum(ev,0); ev/=ev.sum()
    return (evec*ev)@evec.conj().T

def cp_ci(k, n, alpha=0.05):
    lo=beta_dist.ppf(alpha/2,k,n-k+1) if k>0 else 0.0
    hi=beta_dist.ppf(1-alpha/2,k+1,n-k) if k<n else 1.0
    return float(lo), float(hi)

def bootstrap_F(tX,tY,tZ, rho_M, n=N_BOOTSTRAP):
    bsF=np.zeros(n)
    for i in range(n):
        def rs(d):
            keys=list(d.keys()); vals=np.array([d[k] for k in keys])
            N=vals.sum(); new=RNG.multinomial(N,vals/N)
            return {k:int(v) for k,v in zip(keys,new)}
        sx=pauli_exp(rs(tX)); sy=pauli_exp(rs(tY)); sz=pauli_exp(rs(tZ))
        bsF[i]=state_fidelity(DensityMatrix(reconstruct(sx,sy,sz)),rho_M)
    return bsF

def compute_metrics(tomo_X, tomo_Y, tomo_Z, n_total, n_kept, rho_M):
    sx=pauli_exp(tomo_X); sy=pauli_exp(tomo_Y); sz=pauli_exp(tomo_Z)
    rho=reconstruct(sx,sy,sz)
    F=float(state_fidelity(DensityMatrix(rho),rho_M))
    p=n_kept/n_total
    ci_p=cp_ci(n_kept,n_total)
    bsF=bootstrap_F(tomo_X,tomo_Y,tomo_Z,rho_M)
    return {
        'F': F, 'F_lo': float(np.percentile(bsF,2.5)),
        'F_hi': float(np.percentile(bsF,97.5)),
        'F_std': float(np.std(bsF)),
        'p_succ': float(p), 'p_lo': ci_p[0], 'p_hi': ci_p[1],
        'n_kept': int(n_kept), 'n_total': int(n_total),
        'bloch': {'sx':float(sx),'sy':float(sy),'sz':float(sz)},
        'bootstrap_F': [float(v) for v in bsF]
    }

# Target state
from qiskit import QuantumCircuit as QC
qcm=QC(1); qcm.u(THETA_MSG,VARPHI_MSG,0.0,0)
rho_M=DensityMatrix(Statevector.from_instruction(qcm))

# =============================================================================
#  STEP 9 — Analyse each condition
# =============================================================================
results = {}

# ── 9a. Unmitigated ───────────────────────────────────────────────────────────
print("\n  Processing: unmitigated ...")
tomo_um = {}
for basis in ['Z','X','Y']:
    kept,nt,nk = postselect_00(raw[f'unmitigated_{basis}'])
    tomo_um[basis]=kept
    if basis=='Z': nt_um,nk_um=nt,nk
results['unmitigated'] = compute_metrics(
    tomo_um['X'],tomo_um['Y'],tomo_um['Z'],nt_um,nk_um,rho_M)

# ── 9b. Dynamical decoupling ──────────────────────────────────────────────────
print("  Processing: dynamical decoupling ...")
tomo_dd={}
for basis in ['Z','X','Y']:
    kept,nt,nk=postselect_00(raw[f'dd_{basis}'])
    tomo_dd[basis]=kept
    if basis=='Z': nt_dd,nk_dd=nt,nk
results['dd'] = compute_metrics(
    tomo_dd['X'],tomo_dd['Y'],tomo_dd['Z'],nt_dd,nk_dd,rho_M)

# ── 9c. Readout mitigation (mthree) ──────────────────────────────────────────
print("  Processing: readout mitigation ...")
if MTHREE_AVAILABLE:
    tomo_ro={}
    nt_ro=0; nk_ro=0
    for basis in ['Z','X','Y']:
        # mthree expects {bitstring: count} keyed on physical qubit ordering
        # We feed the raw joint counts and let mthree correct them
        raw_counts = raw[f'unmitigated_{basis}']
        # mthree works on the full qubit-ordered bitstring; we need to
        # separate into per-register counts for correction, then recombine.
        # Simplest approach: correct the crTomo register only (1-qubit matrix)
        # using the physical qubit that crTomo maps to.
        crT_counts = {}
        crR_counts_for_ps = {}
        crG_counts_for_ps = {}
        joint_for_ps = defaultdict(int)
        for bs, cnt in raw_counts.items():
            crT,crG,crR = parse_bs(bs)
            crT_counts[crT]  = crT_counts.get(crT,0)+cnt
            joint_for_ps[(crT,crG,crR)] += cnt

        # Apply mthree to crTomo bit (qubit 0 in used_qubits = C register)
        c_phys = used_qubits[0]   # physical qubit of C register
        quasi  = mit.apply_correction(crT_counts, [c_phys])
        corr   = quasi.nearest_probability_distribution()
        # Scale back to counts
        total_kept=0
        kept_ro=defaultdict(int)
        for bs,cnt in raw_counts.items():
            crT,crG,crR=parse_bs(bs)
            if crR=='0' and crG=='0':
                # reweight crTomo bit by correction factor
                p_raw  = crT_counts.get(crT,1)/sum(crT_counts.values())
                p_corr = corr.get(crT,p_raw)
                scale  = p_corr/p_raw if p_raw>0 else 1.0
                w      = int(round(cnt*scale))
                kept_ro[crT]+=w; total_kept+=w

        nt_raw = sum(raw_counts.values())
        tomo_ro[basis]=dict(kept_ro)
        if basis=='Z': nt_ro,nk_ro=nt_raw,total_kept
    results['readout'] = compute_metrics(
        tomo_ro['X'],tomo_ro['Y'],tomo_ro['Z'],nt_ro,nk_ro,rho_M)
else:
    results['readout'] = None

# ── 9d. ZNE — manual gate folding + linear extrapolation ─────────────────────
print("  Processing: ZNE (manual gate folding) ...")
F_at_lambda = {}
p_at_lambda = {}
for lam in ZNE_FACTORS:
    tomo_zne={}
    for basis in ['Z','X','Y']:
        key = 'unmitigated' if lam==1 else f'zne_{lam}'
        kept,nt,nk=postselect_00(raw[f'{key}_{basis}'])
        tomo_zne[basis]=kept
        if basis=='Z': nt_z,nk_z=nt,nk
    sx=pauli_exp(tomo_zne['X']); sy=pauli_exp(tomo_zne['Y']); sz=pauli_exp(tomo_zne['Z'])
    rho=reconstruct(sx,sy,sz)
    F_at_lambda[lam]=float(state_fidelity(DensityMatrix(rho),rho_M))
    p_at_lambda[lam]=nk_z/nt_z

# Linear extrapolation to λ=0
lambdas = np.array(ZNE_FACTORS, dtype=float)
F_vals  = np.array([F_at_lambda[l] for l in ZNE_FACTORS])
p_vals  = np.array([p_at_lambda[l] for l in ZNE_FACTORS])

# Fit F(λ) = a + b*λ → F(0) = a
coeffs_F = np.polyfit(lambdas, F_vals, deg=1)
F_zne    = float(coeffs_F[1])   # intercept = F at λ=0

coeffs_p = np.polyfit(lambdas, p_vals, deg=1)
p_zne    = float(np.clip(coeffs_p[1], 0, 1))

# Bootstrap ZNE CI by resampling each λ-point
bsF_zne = np.zeros(N_BOOTSTRAP)
for i in range(N_BOOTSTRAP):
    F_b=[]
    for lam in ZNE_FACTORS:
        key='unmitigated' if lam==1 else f'zne_{lam}'
        tomo_b={}
        for basis in ['Z','X','Y']:
            kept,_,_=postselect_00(raw[f'{key}_{basis}'])
            keys=list(kept.keys()); vals=np.array([kept[k] for k in keys])
            if vals.sum()==0: tomo_b[basis]={'0':1,'1':1}; continue
            new=RNG.multinomial(vals.sum(),vals/vals.sum())
            tomo_b[basis]={k:int(v) for k,v in zip(keys,new)}
        sx=pauli_exp(tomo_b['X']); sy=pauli_exp(tomo_b['Y']); sz=pauli_exp(tomo_b['Z'])
        rho_b=reconstruct(sx,sy,sz)
        F_b.append(float(state_fidelity(DensityMatrix(rho_b),rho_M)))
    c=np.polyfit(lambdas,F_b,deg=1)
    bsF_zne[i]=c[1]

results['zne'] = {
    'F': F_zne,
    'F_lo': float(np.percentile(bsF_zne,2.5)),
    'F_hi': float(np.percentile(bsF_zne,97.5)),
    'F_std': float(np.std(bsF_zne)),
    'p_succ': p_zne,
    'F_at_lambda': {str(l): F_at_lambda[l] for l in ZNE_FACTORS},
    'p_at_lambda': {str(l): p_at_lambda[l] for l in ZNE_FACTORS},
    'fit_coeffs': coeffs_F.tolist(),
    'bootstrap_F': [float(v) for v in bsF_zne],
    'lambdas': ZNE_FACTORS,
}

# =============================================================================
#  STEP 10 — Summary
# =============================================================================
print("\n" + "=" * 65)
print("  RESULTS — Mitigated vs Unmitigated")
print("=" * 65)
print(f"\n  {'Condition':20s}  {'F(ρY,ρM)':>10}  {'95% CI':>22}  "
      f"{'p_succ':>8}  {'n_kept':>8}")
print("  " + "-" * 75)

cond_labels = [
    ('unmitigated', 'Unmitigated'),
    ('dd',          'Dynamical Dec. (XX)'),
    ('readout',     'Readout Mit. (mthree)'),
    ('zne',         'ZNE (λ=1,3,5 linear)'),
]
for key, label in cond_labels:
    r = results[key]
    if r is None:
        print(f"  {label:20s}  {'N/A':>10}")
        continue
    ci = f"[{r['F_lo']:.4f}, {r['F_hi']:.4f}]"
    nk = r.get('n_kept', '—')
    print(f"  {label:20s}  {r['F']:>10.4f}  {ci:>22}  "
          f"{r['p_succ']:>8.4f}  {nk!s:>8}")

um_F = results['unmitigated']['F']
print(f"\n  Improvement over unmitigated:")
for key, label in cond_labels[1:]:
    r = results[key]
    if r is None: continue
    delta = r['F'] - um_F
    sign  = '+' if delta >= 0 else ''
    print(f"    {label:20s}: ΔF = {sign}{delta:.4f}")

# =============================================================================
#  STEP 11 — Save
# =============================================================================
class _Enc(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, np.ndarray): return o.tolist()
        return super().default(o)

payload = {
    "metadata": {
        "backend": BACKEND, "shots": SHOTS,
        "timestamp": datetime.datetime.utcnow().isoformat()+"Z",
        "theta_msg": THETA_MSG, "varphi_msg": VARPHI_MSG,
        "dd_sequence": DD_SEQUENCE,
        "zne_factors": ZNE_FACTORS,
        "mthree_available": MTHREE_AVAILABLE,
    },
    "results": results,
    "raw_counts": raw,
}

with open("error_mitigation_results.json","w") as f:
    json.dump(payload, f, indent=2, cls=_Enc)
print("\n[✓] Results saved to error_mitigation_results.json")
print("[✓] Run plot_error_mitigation.py for paper figures.")

qiskit_runtime_service._discover_account:WARNING:2026-03-22 12:23:53,128: Loading account with the given token. A saved account will not be used.


  Error Mitigation Baseline — ibm_torino


qiskit_runtime_service.__init__:WARNING:2026-03-22 12:23:56,625: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: CTCs. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-03-22 12:23:56,626: Using instance: CTCs, plan: open



[✓] Connected to ibm_torino

  Transpiling 12 circuits ...
    unmitigated  Z  λ=1  DD=N  depth=  85  2q= 32
    dd           Z  λ=1  DD=Y  depth=  85  2q= 32
    zne_3        Z  λ=3  DD=N  depth=  85  2q= 32
    zne_5        Z  λ=5  DD=N  depth=  85  2q= 32
    unmitigated  X  λ=1  DD=N  depth=  88  2q= 32
    dd           X  λ=1  DD=Y  depth=  88  2q= 32
    zne_3        X  λ=3  DD=N  depth=  88  2q= 32
    zne_5        X  λ=5  DD=N  depth=  88  2q= 32
    unmitigated  Y  λ=1  DD=N  depth=  84  2q= 32
    dd           Y  λ=1  DD=Y  depth=  84  2q= 32
    zne_3        Y  λ=3  DD=N  depth=  84  2q= 32
    zne_5        Y  λ=5  DD=N  depth=  84  2q= 32

  Physical qubits used: [80, 81, 92, 96, 97, 98, 99]

  Submitting 12 circuits × 10000 shots (Batch) ...
    unmitigated_Z: job_id=d701f7itnsts73etokrg
    dd_Z: job_id=d701f7ov5rlc73f5hfm0
    zne_3_Z: job_id=d701f7qtnsts73etoksg
    zne_5_Z: job_id=d701f8469uic73cjr42g
    unmitigated_X: job_id=d701f8469uic73cjr430
    dd_X: job_id=d70